In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

for candidate in (Path.cwd(), Path.cwd().parent):
    pymrm_src = candidate / "pymrm" / "src"
    if pymrm_src.exists() and str(pymrm_src) not in sys.path:
        sys.path.insert(0, str(pymrm_src))

from pymrm import (
    NumJac,
    construct_coefficient_matrix,
    construct_convflux_upwind,
    construct_div,
    construct_grad,
    newton,
    non_uniform_grid,
)


In [ ]:

@dataclass
class ModelConfig:
    length: float = 1.0 # length
    P: float = 0.002 # permeability
    v_ret: float = 1e-1 # velocity of retentate
    v_perm: float = 1e-1 # velocity of permeate
    
    R_ret: float = 5e-3 # radius of retentate
    R_perm: float = R_ret + 5e-4 # radius of permeate
    R_out: float = 7e-3 # outer radius
    
    n_z: int = 100 # number of grid oints in the axial direction
    n_r_perm: int = 30 # number of grid points in the radial direction of permeate
    n_r_ret: int = 30 # number of grid points in the axial direction of retentate
    n_c: int = 5 # number of components
    
    d_ax: float = 2.0e-5 
    particle_radius: float = 1.0e-3
    particle_diffusivity: np.ndarray = field(default_factory=lambda: np.array([2.0e-6, 1.5e-6, 1.5e-6, 1.5e-6, 1.5e-6]))
    k_1: float = 3.0
    k_2: float = 1.0
    k_3: float = 2.0
    k_4: float = 1.0
    eps_s: float = 0.35
    inlet_concentration: np.ndarray = field(default_factory=lambda: np.array([1.0, 3.0, 0.0, 0.0, 0.0]))
    film_pair_coefficients: np.ndarray = field(
        default_factory=lambda: np.array(
            [
                [np.inf, 2.0e-3, 2.0e-3, 2.0e-3, 2.0e-3],
                [2.0e-3, np.inf, 4.0e-3, 3.0e-3, 2.5e-3],
                [2.0e-3, 4.0e-3, np.inf, 3.5e-3, 2.0e-3],
                [2.0e-3, 3.0e-3, 3.5e-3, np.inf, 2.2e-3],
                [2.0e-3, 2.5e-3, 2.0e-3, 2.2e-3, np.inf],
                
            ]
        )
    )
    tol: float = 1.0e-8
    maxfev: int = 20

    @property
    def external_area(self):
        return 3.0 * self.eps_s / self.particle_radius

    @property
    def kma(self):
        return self.film_pair_coefficients * self.external_area
    
    @property
    def a_ret(self):
        return 2.0 / self.R_ret
    
    @property
    def a_perm(self):
        return (2.0 * self.R_ret) / (self.R_out**2 - self.R_perm**2)

cfg = ModelConfig()

pd.Series(
    {
        "axial cells": cfg.n_z,
        "particle radial cells": [cfg.n_r_ret, cfg.n_r_perm],
        "reactor length [m]": cfg.length,
        "velocity [m/s]": [cfg.v_perm, cfg.v_ret],
        "particle radius [mm]": 1e3 * cfg.particle_radius,
        "solid holdup [-]": cfg.eps_s,
        "external area [1/m]": cfg.external_area,
        "reaction rate constant [1/s]": [cfg.k_1, cfg.k_2, cfg.k_3, cfg.k_4], 
    },
    name="settings",
)

In [ ]:
class ReactorParticleMaxwellStefanModel:
    species_labels = ("A", "B", "C", "D", "E") # CO2, H2, CH3OH, H2O, DMC
    stoich = np.array([[-1.0, -3.0, 1.0, 1.0, 0.0], # r1: A + 3B <-> C + D
                      [-1.0, 0.0, -2.0, 1.0, 1.0]]) # r2: A + 2C <-> D + E 

    def __init__(self, cfg: ModelConfig):
        self.cfg = cfg
        self.gas_shape = (cfg.n_z, cfg.n_c)
        self.boundary_shape = (cfg.n_z, 1, cfg.n_c)
        self.particle_shape = (cfg.n_z, cfg.n_r_ret, cfg.n_c)
        self.shape = (cfg.n_z, cfg.n_r_ret + 3, cfg.n_c)
        self._build_grids()
        self._build_operators()
        self.numjac = NumJac(self.shape, axes_diagonals=[0], axes_blocks=[1, 2])
        self.u0 = self.initial_state()
        self.u = self.u0.copy()

    def _build_grids(self):
        cfg = self.cfg
        self.z_f = np.linspace(0.0, cfg.length, cfg.n_z + 1)
        self.z_c = 0.5 * (self.z_f[:-1] + self.z_f[1:])
        self.r_f = non_uniform_grid(0.0, cfg.particle_radius, cfg.n_r_ret + 1, 0.15 * cfg.particle_radius, 0.75)
        self.r_c = 0.5 * (self.r_f[:-1] + self.r_f[1:])
        self.volume_weights = np.diff(self.r_f**3) / self.r_f[-1] ** 3

    def _build_operators(self):
        cfg = self.cfg
        inlet_flux = cfg.v_ret * cfg.inlet_concentration
        bc_axial = (
            {"a": cfg.d_ax, "b": cfg.v_ret, "d": inlet_flux},
            {"a": 1.0, "b": 0.0, "d": 0.0},
        )
        grad_mat, grad_bc = construct_grad(self.gas_shape, self.z_f, self.z_c, bc=bc_axial, axis=0)
        conv_mat, conv_bc = construct_convflux_upwind(self.gas_shape, self.z_f, self.z_c, bc=bc_axial, v=cfg.v_ret, axis=0)
        div_mat = construct_div(self.gas_shape, self.z_f, axis=0)
        d_ax_mat = construct_coefficient_matrix(cfg.d_ax, self.gas_shape, axis=0)
        self.gas_transport_mat = div_mat @ (conv_mat - d_ax_mat @ grad_mat)
        self.gas_transport_const = div_mat @ (conv_bc - d_ax_mat @ grad_bc)

        bc_particle = (
            {"a": 1.0, "b": 0.0, "d": 0.0},
            {"a": 0.0, "b": 1.0, "d": 1.0},
        )
        grad_p_mat, _, grad_p_bc = construct_grad(
            self.particle_shape,
            self.r_f,
            self.r_c,
            bc=bc_particle,
            axis=1,
            shapes_d=(None, self.boundary_shape),
        )
        div_p_mat = construct_div(self.particle_shape, self.r_f, nu=2, axis=1)
        d_p = cfg.particle_diffusivity.reshape(1, 1, cfg.n_c)
        d_p_mat = construct_coefficient_matrix(d_p, self.particle_shape, axis=1)
        flux_p_mat = -d_p_mat @ grad_p_mat
        flux_p_bc = -d_p_mat @ grad_p_bc
        self.particle_diffusion_mat = div_p_mat @ flux_p_mat
        self.particle_boundary_mat = div_p_mat @ flux_p_bc

        face_shape = (cfg.n_z, cfg.n_r_ret + 1, cfg.n_c)
        outer_face_rows = (
            face_shape[2] * face_shape[1] * np.arange(face_shape[0]).reshape((-1, 1))
            + face_shape[2] * cfg.n_r_ret
            + np.arange(face_shape[2]).reshape((1, -1))
        ).ravel()
        self.particle_apparent_mat = (3.0 / cfg.particle_radius) * flux_p_mat[outer_face_rows, :]
        self.particle_apparent_bc_mat = (3.0 / cfg.particle_radius) * flux_p_bc[outer_face_rows, :]

        inlet_flux_m = cfg.v_perm * np.zeros(cfg.n_c) 
        bc_permeate = (
            {"a": 0.0, "b": cfg.v_perm, "d": inlet_flux_m}, 
            {"a": 1.0, "b": 0.0, "d": 0.0},
        )
        
        conv_mat_m, conv_bc_m = construct_convflux_upwind(
            self.gas_shape, self.z_f, self.z_c, bc=bc_permeate, v=cfg.v_perm, axis=0
        )
        div_mat_m = construct_div(self.gas_shape, self.z_f, axis=0)
        
        self.perm_transport_mat = div_mat_m @ conv_mat_m
        self.perm_transport_const = div_mat_m @ conv_bc_m

    def initial_state(self):
        u = np.zeros(self.shape)
        u[:, :-1, :] = self.cfg.inlet_concentration.reshape(1, 1, self.cfg.n_c)
        u[:, -1, :] = 0.0
        return u

    def split_state(self, u):
        u = u.reshape(self.shape)
        return u[:, 0, :], u[:, 1, :], u[:, 2:-1, :], u[:, -1, :]

    def particle_source(self, c_p):
        r1 = (self.cfg.k_1 * c_p[..., 0] * (c_p[..., 1]**3) - self.cfg.k_2 * c_p[..., 2] * c_p[..., 3])
        r2 = (self.cfg.k_3 * c_p[..., 0] * (c_p[..., 2]**2) - self.cfg.k_4 * c_p[..., 4] * c_p[..., 3])
        
        rates = np.stack([r1, r2], axis=-1)
        return np.einsum('zrc,co->zro', rates, self.stoich)

    def particle_average_source(self, c_p):
        source = self.particle_source(c_p)
        return np.sum(source * self.volume_weights.reshape(1, -1, 1), axis=1)

    def particle_apparent_source(self, c_p, c_b):
        c_p_vec = c_p.reshape(-1, 1)
        c_b_vec = c_b[:, None, :].reshape(-1, 1)
        return (
            self.particle_apparent_mat @ c_p_vec
            + self.particle_apparent_bc_mat @ c_b_vec
        ).reshape(self.gas_shape)

    def maxwell_stefan_film_residual(self, c_g, c_b, source_reactor):
        c_mid = 0.5 * (c_g + c_b)
        y_mid = c_mid / np.maximum(np.sum(c_mid, axis=-1, keepdims=True), 1.0e-30)
        correction = np.zeros_like(c_g)
        for i in range(self.cfg.n_c):
            for j in range(self.cfg.n_c):
                if i != j:
                    correction[:, i] += (
                        y_mid[:, j] * source_reactor[:, i]
                        - y_mid[:, i] * source_reactor[:, j]
                    ) / self.cfg.kma[i, j]
        return c_b - c_g - correction

    def residual_values(self, u):
        c_g, c_b, c_p, c_m = self.split_state(u)
        source_particle = self.particle_source(c_p)
        source_reactor = self.cfg.eps_s * self.particle_apparent_source(c_p, c_b)
        membrane_flux_density = self.cfg.P * (c_g - c_m)
        residual = np.zeros_like(u)
        residual[:, 0, :] = (
            self.gas_transport_const
            + self.gas_transport_mat @ c_g.reshape(-1, 1)
        ).reshape(self.gas_shape) - source_reactor + (self.cfg.a_ret * membrane_flux_density)
        residual[:, 1, :] = self.maxwell_stefan_film_residual(c_g, c_b, source_reactor)
        residual[:, 2:-1, :] = (
            self.particle_diffusion_mat @ c_p.reshape(-1, 1)
            + self.particle_boundary_mat @ c_b[:, None, :].reshape(-1, 1)
        ).reshape(self.particle_shape) - source_particle
        residual[:, -1, :] = (
            self.perm_transport_const + self.perm_transport_mat @ c_m.reshape(-1, 1)
        ).reshape(self.gas_shape) - (self.cfg.a_perm * membrane_flux_density)
        return residual

    def residual(self, u):
        u = u.reshape(self.shape)
        residual = self.residual_values(u)
        residual, jac = self.numjac(self.residual_values, u, f_value=residual)
        return residual.ravel(), jac

    def solve(self):
        result = newton(self.residual, self.u, tol=self.cfg.tol, maxfev=self.cfg.maxfev, solver="spsolve")
        self.u = result.x.reshape(self.shape)
        self.result = result
        return result

    def fields(self):
        return self.split_state(self.u)

    def effectiveness_profile(self):
        _c_g, c_b, c_p, c_m = self.fields()
        rate_apparent = self.particle_apparent_source(c_p, c_b)
        r1_surface = (self.cfg.k_1 * c_p[..., 0] * (c_p[..., 1]**3) - self.cfg.k_2 * c_p[..., 2] * c_p[..., 3])
        r2_surface = (self.cfg.k_3 * c_p[..., 0] * (c_p[..., 2]**2) - self.cfg.k_4 * c_p[..., 4] * c_p[..., 3])
        r1_apparent = -rate_apparent[:, 1] / 3.0
        r2_apparent = -rate_apparent[:, 4]
        eta_r1 = -r1_apparent / np.maximum(r1_surface, 1.0e-30)
        eta_r2 = -r2_apparent / np.maximum(r2_surface, 1.0e-30)
        return eta_r1, eta_r2
    
model = ReactorParticleMaxwellStefanModel(cfg)
result = model.solve()
c_g, c_b, c_p, c_m = model.fields()
source_reactor = cfg.eps_s * model.particle_apparent_source(c_p, c_b)

pd.Series(
    {
        "success": result.success,
        "message": result.message,
        "Newton iterations": result.nit,
        "final residual norm": np.linalg.norm(result.fun, ord=np.inf),
        "minimum concentration [mol/m3]": np.min(model.u),
        "outlet CO2 conversion": 1.0 - c_g[-1, 0] / c_g[0, 0],
        "Outlet H2 conversion": 1.0 - c_g[-1, 1] / c_g[0, 1],
        "outlet gas c_A [mol/m3]": c_g[-1, 0],
        "outlet boundary c_A [mol/m3]": c_b[-1, 0],
    },
    name="solve summary",
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=True)

# Left: gas bulk + boundary layer concentrations
for i, label in enumerate(model.species_labels):
    axes[0].plot(model.z_c, c_g[:, i], label=f"gas {label}")
    axes[0].plot(model.z_c, c_b[:, i], "--", label=f"boundary {label}")
axes[0].set_xlabel("z [m]")
axes[0].set_ylabel("concentration [mol m$^{-3}$]")
axes[0].set_title("Retentate bulk & boundary layer")
axes[0].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), ncol=1, fontsize=8)
axes[0].grid(True)

# Middle: cb - cg (external mass transfer driving force)
for i, label in enumerate(model.species_labels):
    axes[1].plot(model.z_c, c_b[:, i] - c_g[:, i], label=label)
axes[1].axhline(0.0, color="k", linewidth=0.8)
axes[1].set_xlabel("z [m]")
axes[1].set_ylabel("$c_b - c_g$ [mol m$^{-3}$]")
axes[1].set_title("External mass transfer driving force")
axes[1].legend()
axes[1].grid(True)

# Right: permeate concentrations + driving force across membrane
for i, label in enumerate(model.species_labels):
    axes[2].plot(model.z_c, c_m[:, i], label=f"permeate {label}")
    axes[2].plot(model.z_c, c_g[:, i] - c_m[:, i], "--", label=f"$\Delta c$ {label}")
axes[2].axhline(0.0, color="k", linewidth=0.8)
axes[2].set_xlabel("z [m]")
axes[2].set_ylabel("concentration [mol m$^{-3}$]")
axes[2].set_title("Permeate & membrane driving force ($c_g - c_m$)")
axes[2].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), ncol=1, fontsize=8)
axes[2].grid(True)

plt.tight_layout()
plt.show()